In [ ]:
!conda install -c conda-forge xgboost -y
!pip install lightgbm

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, average_precision_score, confusion_matrix
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import lightgbm as lgb
import time

In [6]:
def evaluate_model(model_name, y_true, y_pred, y_prob, train_time):
    """Calculates and prints model metrics for imbalanced data."""
    pr_auc = average_precision_score(y_true, y_prob)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    conf_matrix = confusion_matrix(y_true, y_pred)

    print(f'---{model_name} Results---')
    print(f'Training Time: {train_time:.2f} seconds')
    print(f'PR-AUC: {pr_auc:.3f}')
    print(f'Precision: {precision:.3f}')
    print(f'Recall: {recall:.3f}')
    print(f'Confusion Matrix: \n{conf_matrix}\n')

In [34]:
print('Loading Matrices from Disk...')
X_train = pd.read_parquet('data_ready/ml_matrices/X_train.parquet').astype({'vault_id': 'int16', 'cluster_id': 'int8'})
X_test = pd.read_parquet('data_ready/ml_matrices/X_test.parquet').astype({'vault_id': 'int16', 'cluster_id': 'int8'})
y_train = pd.read_parquet('data_ready/ml_matrices/y_train.parquet').squeeze()
y_test = pd.read_parquet('data_ready/ml_matrices/y_test.parquet').squeeze()

print(f'X_train shape: {X_train.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'y_test shape: {y_test.shape}')

Loading Matrices from Disk...
X_train shape: (765101, 89)
y_train shape: (765101,)
X_test shape: (155648, 89)
y_test shape: (155648,)


In [35]:
X_train

,capacity_bytes,smart_1_raw,smart_5_raw,smart_9_raw,smart_194_raw,smart_197_raw,smart_2_raw,smart_3_raw,smart_4_raw,smart_7_raw,...,pod_id,is_legacy_format,smart_71_raw,smart_90_raw,cluster_id,pod_slot_num,smart_82_raw,smart_27_raw,smart_211_raw,smart_212_raw
0,8001563222016,111314472.0,22513.0,72366.0,38.0,0.0,0.0,0.0,21.0,2.108343e+09,...,7,0,0.0,0.0,0,0,0.0,0.0,0.0,0.0
1,8001563222016,136209792.0,11176.0,65025.0,42.0,0.0,0.0,0.0,40.0,9.516666e+08,...,1,0,0.0,0.0,0,0,0.0,0.0,0.0,0.0
2,12000138625024,40867320.0,0.0,33885.0,30.0,0.0,0.0,0.0,12.0,6.462630e+08,...,12,0,0.0,0.0,0,50,0.0,0.0,0.0,0.0
3,12000138625024,0.0,10.0,49440.0,29.0,100.0,0.0,419.0,20.0,0.000000e+00,...,12,0,0.0,0.0,0,4,0.0,0.0,0.0,0.0
4,16000900661248,0.0,0.0,19241.0,39.0,0.0,100.0,327.0,12.0,0.000000e+00,...,0,0,0.0,0.0,0,33,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
765096,14000519643136,94608304.0,0.0,37597.0,30.0,0.0,0.0,0.0,7.0,8.098784e+08,...,8,0,0.0,0.0,0,12,0.0,0.0,0.0,0.0
765097,14000519643136,0.0,0.0,36506.0,28.0,0.0,0.0,7848.0,32.0,0.000000e+00,...,9,0,0.0,0.0,0,57,0.0,0.0,0.0,0.0
765098,12000138625024,88246200.0,0.0,46125.0,33.0,0.0,0.0,0.0,11.0,5.087589e+08,...,19,0,0.0,0.0,0,44,0.0,0.0,0.0,0.0
765099,16000900661248,0.0,0.0,22059.0,35.0,0.0,100.0,347.0,9.0,0.000000e+00,...,10,0,0.0,0.0,40,44,0.0,0.0,0.0,0.0


# Phase 1: Binary Classification (Will the drive fail?)

## Random Forest Baseline

### Random Forest builds many independent trees, it is robust to overfitting and doesn't require much hyperparameter tuning. Because of this the results from this model will get a "decent" baseline to compare other models to.

In [20]:
print('Training Random Forest Baseline...')
start_time = time.time()
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=6740, n_jobs=-1)
rf_model.fit(X_train, y_train)

rf_time = time.time() - start_time

rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

evaluate_model('Random Forest', y_test, rf_pred, rf_prob, rf_time)

Training Random Forest Baseline...
---Random Forest Results---
Training Time: 98.82 seconds
PR-AUC: 0.710
Precision: 0.913
Recall: 0.532
Confusion Matrix: 
[[154655     48]
 [   442    503]]



## Gradient Boosting Comparison

### Comparing the training/testing speed, memory efficiency, and predictive ability of two types of gradient boosting models. 

#### XGBoost

In [31]:
print('Training XGBoost...')
start_time = time.time()

weight = (y_train == 0).sum() / (y_train == 1).sum()
xgb_model = xgb.XGBClassifier(
    scale_pos_weight=weight,
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=6740,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)
xgb_time = time.time() - start_time
xgb_pred = xgb_model.predict(X_test)
xgb_prob = xgb_model.predict_proba(X_test)[:, 1]
evaluate_model('XGBoost', y_test, xgb_pred, xgb_prob, xgb_time)

Training XGBoost...
---XGBoost Results---
Training Time: 8.21 seconds
PR-AUC: 0.691
Precision: 0.156
Recall: 0.870
Confusion Matrix: 
[[150239   4464]
 [   123    822]]



#### LightGBM

In [32]:
print('Training LightGBM...')
start_time = time.time()

lgb_model = lgb.LGBMClassifier(
    class_weight='balanced',
    n_estimators=100,
    random_state=6740,
    n_jobs=-1
)

lgb_model.fit(X_train, y_train)
lgb_time = time.time() - start_time
lgb_pred = lgb_model.predict(X_test)
lgb_prob = lgb_model.predict_proba(X_test)[:, 1]
evaluate_model('LightGBM', y_test, lgb_pred, lgb_prob, lgb_time)

Training LightGBM
[LightGBM] [Info] Number of positive: 5922, number of negative: 759179
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.081505 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9211
[LightGBM] [Info] Number of data points in the train set: 765101, number of used features: 69
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
---LightGBM Results---
Training Time: 6.05 seconds
PR-AUC: 0.703
Precision: 0.185
Recall: 0.871
Confusion Matrix: 
[[151078   3625]
 [   122    823]]



# Phase 2: Survival Analysis (When will the drive fail?)

## Cox Proportional Hazards Baseline

### Assumes a linear combination of SMART stats increases the base "hazard rate". Provides interpretable coefficients.

## Non-Linear Survival Models

### Does not assume a linear relationship. 

# Phase 3: Anomaly Detection (Is this drive an outlier?)

## Isolation Forest

## Autoencoder